# GPT Language Models: Analysis & Explanation

Welcome to the **Mini-GPT & NanoGPT Project Explanation Notebook**. This guide breaks down the core concepts, dataset, model architectures, and text generation process of our custom-trained GPT models.

## Project Overview
We have two primary implementations in this project:
1. **Custom Mini-GPT**: A modular, simplified transformer decoder.
2. **Improved NanoGPT**: A feature-rich model matching the original Andrej Karpathy design, featuring weight tying, Flash Attention, gradient clipping, and cosine learning rate decay with warmup.

Let's dive into the code and architecture!

## 1. Dataset & Tokenization

Our model is trained on the character-level Shakespeare dataset. Instead of subwords (like BPE in GPT-2/3/4), this model tokenizes text character-by-character. This means the vocabulary size is very small (~65 unique characters), making it lightweight and fast to train.

In [ ]:
import torch
from dataset import download_dataset, load_dataset

# Download the Shakespeare text dataset if not already done
download_dataset()

# Load data, vocabulary, and encoding/decoding helper functions
train_data, val_data, vocab_size, encode, decode = load_dataset()

print(f"Vocabulary size: {vocab_size}")
print(f"First 50 characters of train dataset:\n{decode(train_data[:50].tolist())}")

## 2. Model Architecture: Mini-GPT vs NanoGPT

Here is how the two implementations compare:

| Feature | Mini-GPT | NanoGPT (Improved) |
|---|---|---|
| **Attention** | Standard Self-Attention | Flash Attention (`F.scaled_dot_product_attention`) |
| **Weight Tying** | No | Yes (Embedding and output LM head weights are shared) |
| **LayerNorm** | Pre-LN | Pre-LN (Custom with optional bias matching original repo) |
| **Optimizer** | Standard AdamW | Configured AdamW (Weight decay disabled on bias/layer-norm parameters) |
| **LR Scheduler** | Constant | Cosine Decay with Warmup |

Let's instantiate our improved NanoGPT model and load the trained parameters from `nano_gpt_model.pt`.

In [ ]:
from nano_gpt_config import model_config, device
from nano_gpt_model import NanoGPT

print(f"Loading model on device: {device}")
print("Model Configuration:")
print(f" - Block size (context length): {model_config.block_size}")
print(f" - Embedding dimension: {model_config.n_embd}")
print(f" - Attention heads: {model_config.n_head}")
print(f" - Number of transformer layers: {model_config.n_layer}")

# Create the NanoGPT model structure
model = NanoGPT(model_config)

# Load trained weights
model.load_state_dict(torch.load("nano_gpt_model.pt", map_location=device))
model = model.to(device)
model.eval()  # Set model to evaluation mode
print("Model weights loaded successfully!")

## 3. Generating Shakespeare-like Text

Now we can feed a starting token (zero token as context) and generate text by sampling from the logits output by the model.

In [ ]:
# Start with context containing character token 0
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Generate 500 characters of text
print("Generating text...\n")
generated_tokens = model.generate(context, max_new_tokens=500)[0].tolist()
generated_text = decode(generated_tokens)

print("-" * 50)
print(generated_text)
print("-" * 50)